# Profilowanie jakościowe - Milestone 1

Cel: sprawdzenie schematu i jakości danych w warstwie bronze za pomocą zapytań SQL w DuckDB.

Dane: pliki Parquet z `data/bronze/` utworzone w notebookach 02. Notebook tylko czyta dane, niczego nie zapisuje.

## 1. Połączenie z DuckDB i widoki

`duckdb.connect()` bez ścieżki tworzy bazę w pamięci, która istnieje tylko na czas działania notebooka.

Na każdy plik Parquet zakładamy widok (`CREATE VIEW`). Widok to zapisane zapytanie, a nie kopia danych: DuckDB czyta plik dopiero przy zapytaniu i tylko te kolumny, których zapytanie potrzebuje. Dzięki temu nie wczytujemy 9 mln wierszy do RAM.

In [1]:
from pathlib import Path
import duckdb

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
BRONZE_DIR = ROOT / 'data' / 'bronze'

TABLES = ['pomiary_2024', 'stacje', 'stanowiska', 'openmeteo_2024']

con = duckdb.connect()

for table in TABLES:
    path = BRONZE_DIR / f'{table}.parquet'
    con.sql(
        f"CREATE OR REPLACE VIEW {table} AS "
        f"SELECT * FROM read_parquet('{path.as_posix()}')"
    )
    print(f'{table:16} {path.stat().st_size / 1_000_000:6.1f} MB')

print(con.sql('SHOW TABLES'))

pomiary_2024       19.2 MB
stacje              0.1 MB
stanowiska          0.1 MB
openmeteo_2024      7.6 MB
┌────────────────┐
│      name      │
│    varchar     │
├────────────────┤
│ openmeteo_2024 │
│ pomiary_2024   │
│ stacje         │
│ stanowiska     │
└────────────────┘



## 2. Schemat tabel

`DESCRIBE SELECT * FROM ...` zwraca nazwy i typy kolumn. DuckDB odczytuje je z metadanych zapisanych w pliku Parquet, bez skanowania danych.

Kolumny `null` i `key` w wyniku pokazują `YES` i `NULL` dla każdej kolumny, bo Parquet nie przechowuje kluczy ani ograniczeń `NOT NULL`. Klucze opisujemy więc pod każdą tabelą, a ich poprawność sprawdzamy w profilowaniu.

### 2.1. `pomiary_2024`

1 wiersz = 1 pomiar jednego stanowiska w jednym okresie (godzina albo doba).

In [2]:
print(con.sql('DESCRIBE SELECT * FROM pomiary_2024'))

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ jednostka        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ wartosc          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ plik_zrodlowy    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod przyrządu pomiarowego: stacja, wskaźnik i czas uśredniania, np. `DsWrocAlWisn-PM10-1g` |
| `kod_stacji` | kod stacji, na której stoi przyrząd |
| `wskaznik` | mierzona substancja, np. `PM10`, `PM2.5`, `NO2`, `O3`, `BaP(PM10)` (benzo(a)piren w pyle PM10) |
| `czas_usredniania` | `1g` - wartość godzinowa, `24g` - wartość dobowa |
| `jednostka` | `ug/m3` dla większości wskaźników, `mg/m3` dla CO, `ng/m3` dla śladowych składników pyłu (np. BaP, As, Cd, Ni) i rtęci |
| `data_czas` | znacznik czasu pomiaru z pliku GIOŚ, bez strefy czasowej |
| `wartosc` | zmierzone stężenie; `NULL` oznacza brak pomiaru w tym okresie |
| `plik_zrodlowy` | plik XLSX z warstwy RAW, z którego pochodzi wiersz |

Klucz: `(kod_stanowiska, data_czas)` - jedno stanowisko powinno mieć jeden pomiar w danym okresie. Czy tak jest, sprawdzamy w profilowaniu.

Jednostki różnią się między wskaźnikami, więc statystyki (min, max, średnie) liczymy zawsze osobno dla każdego wskaźnika.

### 2.2. `stacje`

1 wiersz = 1 stacja z metadanych GIOŚ. Arkusz obejmuje wszystkie stacje w historii, także zamknięte przed 2024 r.

In [3]:
print(con.sql('DESCRIBE SELECT * FROM stacje'))

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_miedzynarodowy │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ nazwa_stacji       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ stary_kod_stacji   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia  │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_stacji         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_obszaru        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ rodzaj_stacji      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, np. `DsWrocAlWisn` (województwo, miejscowość, ulica) |
| `kod_miedzynarodowy` | kod stacji w europejskiej bazie jakości powietrza, np. `PL0810A`; nie każda stacja go ma |
| `nazwa_stacji` | nazwa opisowa |
| `stary_kod_stacji` | poprzedni kod, jeśli stacja go zmieniła |
| `data_uruchomienia`, `data_zamkniecia` | okres działania; pusta data zamknięcia - stacja działa |
| `typ_stacji` | czego wpływ ma mierzyć: `tło`, `komunikacyjna` (ruch drogowy), `przemysłowa` |
| `typ_obszaru` | `miejski`, `podmiejski`, `pozamiejski` |
| `rodzaj_stacji` | rodzaj obiektu, np. `kontenerowa stacjonarna`, `mobilna`, `w budynku` |
| `wojewodztwo`, `miejscowosc`, `adres` | lokalizacja administracyjna |
| `szerokosc`, `dlugosc` | współrzędne WGS84 |

Klucz: `kod_stacji`.

### 2.3. `stanowiska`

1 wiersz = 1 stanowisko, czyli jeden przyrząd mierzący jeden wskaźnik na jednej stacji.

In [4]:
print(con.sql('DESCRIBE SELECT * FROM stanowiska'))

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stanowiska    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ kod_stacji        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ wskaznik_nazwa    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ czas_usredniania  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ typ_pomiaru       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_uruchomienia │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ data_zamkniecia   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ wojewodztwo       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ strefa    

| Kolumna | Znaczenie |
|---|---|
| `kod_stanowiska` | kod stanowiska, ten sam co w `pomiary_2024` |
| `kod_stacji` | stacja, na której stoi stanowisko |
| `wskaznik`, `wskaznik_nazwa` | kod i pełna nazwa wskaźnika, np. `PM2.5` / `pył zawieszony PM2.5` |
| `czas_usredniania` | zapisany słownie: `1-godzinny`, `24-godzinny`, `2-tygodniowy`, `miesięczny` |
| `typ_pomiaru` | `automatyczny` (analizator, wynik na bieżąco) albo `manualny` (próbka pobrana na filtr i zważona lub zbadana w laboratorium) |
| `data_uruchomienia`, `data_zamkniecia` | okres działania stanowiska |
| `wojewodztwo` | województwo |
| `strefa` | strefa oceny jakości powietrza, np. aglomeracja albo pozostały obszar województwa |

Klucz: `kod_stanowiska`.

Czas uśredniania jest zapisany inaczej niż w `pomiary_2024` (`1-godzinny` zamiast `1g`). Dlatego pomiary łączymy ze stanowiskami po `kod_stanowiska`, a nie po parze wskaźnik i czas uśredniania.

### 2.4. `openmeteo_2024`

1 wiersz = pogoda dla 1 stacji mierzącej PM2.5 w 1 godzinie.

In [5]:
print(con.sql('DESCRIBE SELECT * FROM openmeteo_2024'))

┌──────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│     column_name      │ column_type │  null   │   key   │ default │  extra  │
│       varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ kod_stacji           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ data_czas_utc        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ temperature_2m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ relative_humidity_2m │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ precipitation        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ surface_pressure     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_speed_10m       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ wind_direction_10m   │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ siatka_szerokosc     │ DOUBLE      │ YES     │ NUL

| Kolumna | Znaczenie |
|---|---|
| `kod_stacji` | kod stacji GIOŚ, wzięty z nazwy pliku JSON |
| `data_czas_utc` | godzina w UTC |
| `temperature_2m` | temperatura powietrza 2 m nad ziemią, °C |
| `relative_humidity_2m` | wilgotność względna 2 m nad ziemią, % |
| `precipitation` | suma opadu z poprzedniej godziny, mm |
| `surface_pressure` | ciśnienie na poziomie terenu (nie przeliczone do poziomu morza), hPa |
| `wind_speed_10m` | prędkość wiatru 10 m nad ziemią, km/h |
| `wind_direction_10m` | kierunek, z którego wieje wiatr, ° (0 = północ) |
| `siatka_szerokosc`, `siatka_dlugosc` | punkt siatki modelu pogodowego najbliższy stacji; to nie są dokładne współrzędne stacji |
| `wysokosc_m` | wysokość terenu n.p.m. przyjęta przez Open-Meteo dla tego punktu |
| `plik_zrodlowy` | plik JSON z warstwy RAW |

Klucz: `(kod_stacji, data_czas_utc)`.

To dane z modelu meteorologicznego (reanaliza), a nie pomiar na stacji GIOŚ.

### 2.5. Relacje między tabelami

```text
pomiary_2024   ──(kod_stanowiska)──►  stanowiska  ──(kod_stacji)──►  stacje
openmeteo_2024 ─────────────────────(kod_stacji)──────────────────►  stacje
pomiary_2024   ◄──(kod_stacji + czas)──►  openmeteo_2024     [Milestone 2]
```

- `pomiary_2024.kod_stanowiska` → `stanowiska.kod_stanowiska`
- `pomiary_2024.kod_stacji`, `stanowiska.kod_stacji`, `openmeteo_2024.kod_stacji` → `stacje.kod_stacji`
- pomiary z pogodą łączą się po stacji i czasie. GIOŚ zapisuje czas bez strefy, a Open-Meteo w UTC, więc zgodność czasu sprawdzamy w profilowaniu przed złączeniem w warstwie silver (Milestone 2).